# The Web Search Tool (Lesson 31)

Follow-along notebook for *The web search tool* in Building with the Claude API.

**The key idea:** web search is a **server tool**. Unlike your own tools or the text editor, you don't implement anything: Claude runs the searches on Anthropic's side and returns the queries, results, and citations in one response.

**Before you start:** an admin must enable web search for your organization in the Claude Console (Settings → Privacy). If it's off, the first API call in this notebook returns an error saying so.

**Run order:** Run All after a kernel restart.

## 1. Setup

In [1]:
import json

from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()
client = Anthropic()
MODEL = "claude-haiku-4-5"

## 2. The schema

Just a stub: a versioned `type`, a `name`, and options. `max_uses` caps how many searches Claude can run in one request, since it may do follow-up searches.

`web_search_20250305` is the basic version the lesson uses and works with Haiku 4.5. Newer versions (`web_search_20260209`, `web_search_20260318`) add *dynamic filtering*, where Claude filters results with code before reading them, but they only work with certain larger models.

In [2]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
}

## 3. Asking a question

`ask()` handles one wrinkle the lesson doesn't mention: a long search turn can end with `stop_reason == "pause_turn"`. That means the server paused mid-turn; you send the conversation back as-is and Claude picks up where it left off. There are no tool results for you to send, because the server ran the tool.

In [3]:
def ask(question, tools, max_tokens=2000, max_continues=3):
    messages = [{"role": "user", "content": question}]
    content = []

    for _ in range(max_continues + 1):
        response = client.messages.create(
            model=MODEL, max_tokens=max_tokens, messages=messages, tools=tools,
        )
        content.extend(response.content)
        if response.stop_reason != "pause_turn":
            break
        messages.append({"role": "assistant", "content": response.content})

    response.content = content   # combined blocks from all continuations
    return response


response = ask("What's the best exercise for gaining leg muscle?", tools=[web_search_schema])
print("stop_reason:", response.stop_reason)
search_usage = response.usage.server_tool_use
print("searches run:", search_usage.web_search_requests if search_usage else 0)

stop_reason: end_turn


AttributeError: 'NoneType' object has no attribute 'web_search_requests'

## 4. What's in the response

Block types you'll see:

| Block | What it holds |
|---|---|
| `text` | Claude's writing; may carry `citations` |
| `server_tool_use` | The exact search query Claude ran |
| `web_search_tool_result` | The results list (each a `web_search_result` with title, URL, page age), or an error |

In [ ]:
for i, block in enumerate(response.content):
    if block.type == "server_tool_use":
        print(f"[{i}] server_tool_use   query = {block.input.get('query')!r}")

    elif block.type == "web_search_tool_result":
        if isinstance(block.content, list):
            print(f"[{i}] web_search_tool_result   {len(block.content)} results")
            for r in block.content[:3]:
                print(f"       - {r.title}  ({r.url})")
        else:
            print(f"[{i}] web_search_tool_result   ERROR: {block.content.error_code}")

    elif block.type == "text":
        n = len(block.citations or [])
        preview = block.text.strip().replace("\n", " ")[:80]
        print(f"[{i}] text ({n} citations)   {preview!r}")

To see one block in full, including the citation objects:

In [ ]:
cited = next((b for b in response.content if b.type == "text" and b.citations), None)
if cited:
    print(cited.model_dump_json(indent=2)[:2000])
else:
    print("No cited text blocks in this response.")

## 5. Rendering for a user

Following the lesson's UI guidance:
- **sources** list at the top
- text rendered normally
- **citations** inline as numbered references, with the source title, URL, and the text that supports the claim

`display(Markdown(...))` renders it as formatted text in the notebook.

In [ ]:
def render_response(response):
    sources = []        # unique search results, in order found
    cite_index = {}     # url -> citation number
    body = []
    notes = []

    for block in response.content:
        if block.type == "web_search_tool_result":
            if isinstance(block.content, list):
                for r in block.content:
                    if r.url not in {s.url for s in sources}:
                        sources.append(r)
            else:
                body.append(f"\n> _Search error: {block.content.error_code}_\n")

        elif block.type == "server_tool_use":
            body.append(f"\n> 🔎 _Searched: {block.input.get('query')}_\n\n")

        elif block.type == "text":
            body.append(block.text)
            for c in block.citations or []:
                if c.url not in cite_index:
                    cite_index[c.url] = len(cite_index) + 1
                    notes.append(f"{cite_index[c.url]}. [{c.title}]({c.url})  \n"
                                 f"   > {c.cited_text.strip()[:200]}")
                body.append(f" <sup>[{cite_index[c.url]}]</sup>")

    out = []
    if sources:
        out.append("**Sources consulted:** " +
                   " · ".join(f"[{s.title[:40]}]({s.url})" for s in sources[:8]))
        out.append("\n---\n")
    out.append("".join(body))
    if notes:
        out.append("\n\n---\n**Citations**\n\n" + "\n".join(notes))
    return "\n".join(out)


display(Markdown(render_response(response)))

## 6. Restricting search domains

`allowed_domains` limits searches to sources you trust. Here, only `nih.gov` (which includes PubMed), so exercise advice is evidence-based rather than from random blogs.

There's also `blocked_domains` for the opposite. You can use one or the other in a request, not both.

In [ ]:
nih_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

response_nih = ask("What's the best exercise for gaining leg muscle?", tools=[nih_search_schema])
display(Markdown(render_response(response_nih)))

Confirm every source really came from the allowed domain:

In [ ]:
from urllib.parse import urlparse

domains = {
    urlparse(r.url).netloc
    for b in response_nih.content if b.type == "web_search_tool_result" and isinstance(b.content, list)
    for r in b.content
}
print(domains)

## 7. When Claude decides *not* to search

Claude only searches when it thinks the question needs it. A timeless question usually gets answered directly, with no search cost.

In [ ]:
response_plain = ask("What is the Pythagorean theorem?", tools=[web_search_schema])
print("searches run:", response_plain.usage.server_tool_use.web_search_requests
      if response_plain.usage.server_tool_use else 0)
display(Markdown(render_response(response_plain)))

## Takeaways
- Server tools run on Anthropic's side: no `run_tool`, no tool results to send back.
- The response is a mix of `text`, `server_tool_use`, and `web_search_tool_result` blocks; citations ride on text blocks.
- `max_uses` caps searches (and cost); `allowed_domains` / `blocked_domains` control sources.
- Handle `pause_turn` by re-sending the conversation, and check `web_search_tool_result` for errors, which arrive inside a normal 200 response.